In [1]:
import pandas as pd
from itertools import combinations

In [2]:
# Load GTFS files
stop_times = pd.read_csv("stop_times.txt")
trips = pd.read_csv("trips.txt")
routes = pd.read_csv("routes.txt")

In [3]:
trips

,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed
0,1,1,11708204,Geary + 33rd Avenue,0,102,217842,1,1
1,1,1,11708205,Geary + 33rd Avenue,0,104,217842,1,1
2,1,1,11708206,Geary + 33rd Avenue,0,106,217842,1,1
3,1,1,11708207,Geary + 33rd Avenue,0,107,217842,1,1
4,1,1,11708208,Geary + 33rd Avenue,0,101,217841,1,1
...,...,...,...,...,...,...,...,...,...
25528,TBUS,3,11741271,Visitacion Valley,1,912,218359,1,1
25529,TBUS,3,11741272,Visitacion Valley,1,1505,218359,1,1
25530,TBUS,3,11741273,Visitacion Valley,1,2909,218359,1,1
25531,TBUS,3,11741274,Visitacion Valley,1,1503,218359,1,1


In [4]:
routes

,route_id,agency_id,route_short_name,route_long_name,route_url,route_desc,route_type,route_color,route_text_color,route_sort_order
0,1,SFMTA,1,CALIFORNIA,https://SFMTA.com/1,5am-12 midnight daily,3,005B95,FFFFFF,NaN
1,12,SFMTA,12,FOLSOM-PACIFIC,https://SFMTA.com/12,6am-10pm daily,3,005B95,FFFFFF,NaN
2,14,SFMTA,14,MISSION,https://SFMTA.com/14,24 hour service daily,3,005B95,FFFFFF,NaN
3,14R,SFMTA,14R,MISSION RAPID,https://SFMTA.com/14R,5am-10pm daily,3,BF2B45,FFFFFF,NaN
4,15,SFMTA,15,BAYVIEW HUNTERS POINT EXPRESS,https://SFMTA.com/15,Weekdays 5am-10pm Weekends 8am-10pm,3,005B95,FFFFFF,NaN
...,...,...,...,...,...,...,...,...,...,...
64,T,SFMTA,T,THIRD,https://SFMTA.com/T,Weekends 8 am-11:30 pm,0,BF2B45,FFFFFF,NaN
65,TBUS,SFMTA,TBUS,THIRD BUS,https://SFMTA.com/TBUS,Weekdays 5am-6am Weekends 5am-8am,3,BF2B45,FFFFFF,NaN
66,FBUS,SFMTA,FBUS,MARKET & WHARVES,http://www.sfmta.com/FBUS,NaN,0,B49A36,000000,NaN
67,L,SFMTA,L,TARAVAL,https://SFMTA.com/L,5am-10 pm daily,0,942D83,FFFFFF,NaN


In [6]:
# Routes of interest
target_routes = ['1', '18', '19', '1X', '2', '22', '24', '28', '29', '30', '31',
       '38', '38R', '48', '49', '54', '56', '66']

In [7]:
# Filter routes
filtered_routes = routes[routes['route_id'].astype(str).isin(target_routes)]

In [8]:
# Merge to get relevant trip_ids
filtered_trips = trips[trips['route_id'].isin(filtered_routes['route_id'])]

In [9]:
# Merge with stop_times to get arrival times
filtered_stop_times = stop_times[stop_times['trip_id'].isin(filtered_trips['trip_id'])]

In [10]:
# Convert times, handling "24:00:00" and beyond
def fix_arrival_time(time_str):
    h, m, s = map(int, time_str.split(':'))
    h = h % 24  # Wrap around hours to fit in 0-23
    return f"{h:02}:{m:02}:{s:02}"

In [11]:
filtered_stop_times['arrival_time'] = filtered_stop_times['arrival_time'].apply(fix_arrival_time)

/tmp/ipykernel_446/2052261029.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_stop_times['arrival_time'] = filtered_stop_times['arrival_time'].apply(fix_arrival_time)


In [12]:
# Convert to datetime
filtered_stop_times['arrival_time'] = pd.to_datetime(filtered_stop_times['arrival_time'], format='%H:%M:%S')

/tmp/ipykernel_446/2698092623.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_stop_times['arrival_time'] = pd.to_datetime(filtered_stop_times['arrival_time'], format='%H:%M:%S')


In [13]:
# Filter for arrivals between 7:00 AM and 8:30 AM
mask = (filtered_stop_times['arrival_time'].dt.time >= pd.to_datetime("07:00:00").time()) & \
       (filtered_stop_times['arrival_time'].dt.time <= pd.to_datetime("08:30:00").time())

filtered_stop_times = filtered_stop_times[mask]

In [14]:
# Sort by trip_id and stop_sequence to maintain order
filtered_stop_times = filtered_stop_times.sort_values(by=['trip_id', 'stop_sequence'])

In [15]:
# List of stop_id_2 values to include
valid_stop_ids_2 = { 5337, 5338, 5339, 5340, 5341, 5342, 5343, 5344, 5351, 5352,
    6971, 6972, 6978, 6979, 3053, 3054, 3055, 3056, 3057, 3058,
    3059, 3060, 3550, 3551, 3552, 3553, 3554, 3557, 3558, 3559,
    3560, 4271, 4272, 4273, 4274, 4275, 4277, 4278, 4279, 7836,
    7837, 3388, 3389, 6142, 6143, 6144, 6145, 6146, 6147, 4293,
    4294, 4295, 4414, 4415, 4421, 4422, 4434, 4435, 4474, 4487,
    4488, 4491, 4492, 4614, 4615, 4633, 4634, 4760, 4761, 6589,
    6590, 6607, 6608, 6609, 6610, 3093, 3957, 4845, 5059, 5060,
    5067, 5080, 5081, 5459, 5464, 5468, 5469, 5472, 5987, 5988,
    5990, 5991, 5992, 6800, 6801, 6806, 6819, 6820, 7476, 7529,
    7857}


In [16]:
time_diffs = []

for trip_id, group in filtered_stop_times.groupby('trip_id'):
    stops = group[['stop_id', 'arrival_time']].values.tolist()
    
    # Get all unique pairs of stops
    for (stop1, time1), (stop2, time2) in combinations(stops, 2):
        if stop2 in valid_stop_ids_2 and time2.time() <= pd.to_datetime("08:30:00").time():  
            # Ensure stop_id_2 is in the valid list and time2 is not later than 8:30
            time_diff = abs(time2 - time1)  # Absolute time difference
            time_diffs.append([trip_id, stop1, stop2, time1, time2, time_diff])


In [17]:
# Convert results into DataFrame
time_diff_df = pd.DataFrame(time_diffs, columns=['trip_id', 'stop_id_1', 'stop_id_2', 'arrival_time_1', 'arrival_time_2', 'time_difference'])

In [18]:
# Sort by time difference (ascending order)
time_diff_df = time_diff_df.sort_values(by='time_difference')

In [19]:
# Display results
print(time_diff_df)

        trip_id  stop_id_1  stop_id_2      arrival_time_1      arrival_time_2  \
20873  11722110       5342       5352 1900-01-01 07:12:42 1900-01-01 07:12:53   
20968  11722111       5342       5352 1900-01-01 07:27:42 1900-01-01 07:27:53   
19138  11722002       5342       5352 1900-01-01 07:12:42 1900-01-01 07:12:53   
19043  11722001       5342       5352 1900-01-01 07:27:42 1900-01-01 07:27:53   
21063  11722112       5342       5352 1900-01-01 07:42:45 1900-01-01 07:42:57   
...         ...        ...        ...                 ...                 ...   
15702  11721635       3023       5341 1900-01-01 07:22:24 1900-01-01 08:23:03   
15696  11721635       3706       5351 1900-01-01 07:22:00 1900-01-01 08:22:47   
15703  11721635       3023       5338 1900-01-01 07:22:24 1900-01-01 08:23:18   
15697  11721635       3706       5341 1900-01-01 07:22:00 1900-01-01 08:23:03   
15698  11721635       3706       5338 1900-01-01 07:22:00 1900-01-01 08:23:18   

      time_difference  
208

In [21]:
time_diff_df['time_difference'] = pd.to_timedelta(time_diff_df['time_difference'])


In [36]:
time_diff_with_routes = pd.merge(time_diff_df, trips, on='trip_id', how='left')


In [37]:
time_diff_with_routes = pd.merge(time_diff_with_routes, routes, on='route_id', how='left')


In [39]:
time_diff_with_routes

,trip_id,stop_id_1,stop_id_2,arrival_time_1,arrival_time_2,time_difference,route_id,service_id,trip_headsign,direction_id,...,bikes_allowed,agency_id,route_short_name,route_long_name,route_url,route_desc,route_type,route_color,route_text_color,route_sort_order
0,11722110,5342,5352,1900-01-01 07:12:42,1900-01-01 07:12:53,0 days 00:00:11,29,3,Baker Beach,1,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
1,11722111,5342,5352,1900-01-01 07:27:42,1900-01-01 07:27:53,0 days 00:00:11,29,3,Baker Beach,1,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
2,11722002,5342,5352,1900-01-01 07:12:42,1900-01-01 07:12:53,0 days 00:00:11,29,2,Baker Beach,1,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
3,11722001,5342,5352,1900-01-01 07:27:42,1900-01-01 07:27:53,0 days 00:00:11,29,2,Baker Beach,1,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
4,11722112,5342,5352,1900-01-01 07:42:45,1900-01-01 07:42:57,0 days 00:00:12,29,3,Baker Beach,1,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45216,11721635,3023,5341,1900-01-01 07:22:24,1900-01-01 08:23:03,0 days 01:00:39,29,1,Paul + Third Street,0,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
45217,11721635,3706,5351,1900-01-01 07:22:00,1900-01-01 08:22:47,0 days 01:00:47,29,1,Paul + Third Street,0,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
45218,11721635,3023,5338,1900-01-01 07:22:24,1900-01-01 08:23:18,0 days 01:00:54,29,1,Paul + Third Street,0,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN
45219,11721635,3706,5341,1900-01-01 07:22:00,1900-01-01 08:23:03,0 days 01:01:03,29,1,Paul + Third Street,0,...,1,SFMTA,29,SUNSET,https://SFMTA.com/29,5am-12 midnight daily,3,005B95,FFFFFF,NaN


In [40]:
average_time_differences3 = time_diff_with_routes.groupby(['stop_id_1', 'stop_id_2', 'route_id'])['time_difference'].mean().reset_index()


In [41]:
average_time_differences3

,stop_id_1,stop_id_2,route_id,time_difference
0,390,3388,28,0 days 00:10:22.842105263
1,390,5459,28,0 days 00:43:51
2,390,5469,28,0 days 00:43:00
3,390,6800,28,0 days 00:41:22
4,390,6806,28,0 days 00:42:12
...,...,...,...,...
3092,8159,5338,29,0 days 00:35:05.785714285
3093,8159,5340,29,0 days 00:33:40.928571428
3094,8159,5341,29,0 days 00:34:52
3095,8159,5344,29,0 days 00:34:08.928571428


In [45]:
# Group by stop_id_1 and stop_id_2 and calculate the average time difference
average_time_differences4 = average_time_differences3.groupby(['stop_id_1', 'stop_id_2', 'route_id'])['time_difference'].mean().reset_index()

In [46]:
average_time_differences4

,stop_id_1,stop_id_2,route_id,time_difference
0,390,3388,28,0 days 00:10:22.842105263
1,390,5459,28,0 days 00:43:51
2,390,5469,28,0 days 00:43:00
3,390,6800,28,0 days 00:41:22
4,390,6806,28,0 days 00:42:12
...,...,...,...,...
3092,8159,5338,29,0 days 00:35:05.785714285
3093,8159,5340,29,0 days 00:33:40.928571428
3094,8159,5341,29,0 days 00:34:52
3095,8159,5344,29,0 days 00:34:08.928571428


In [25]:
# Save the results
average_time_differences.to_csv("stop ids with time diff", index=False)

In [30]:
# Extract unique stop IDs from stop_id_1
unique_stop_ids1 = average_time_differences['stop_id_1'].unique().tolist()

In [28]:
stop =pd.read_csv("stops.txt")

In [29]:
stop

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url
0,390,10390,19th Avenue & Holloway St,,37.721190,-122.475153,,https://SFMTA.com/10390
1,913,10913,Dublin St & La Grande Ave,,37.719192,-122.425802,,https://SFMTA.com/10913
2,3016,13016,3rd St & 4th St,,37.772618,-122.389786,,https://SFMTA.com/13016
3,3018,13018,Bacon St & San Bruno Ave,,37.727859,-122.402994,,https://SFMTA.com/13018
4,3019,13019,Bacon St & San Bruno Ave,,37.727645,-122.403269,,https://SFMTA.com/13019
...,...,...,...,...,...,...,...,...
3271,8155,18155,Church St & 26th St,,37.748416,-122.427060,,https://SFMTA.com/18155
3272,8156,18156,Church St & 26th St,,37.748603,-122.427261,,https://SFMTA.com/18156
3273,8157,18157,Church St & 28th St,,37.745225,-122.426737,,https://SFMTA.com/18157
3274,8158,18158,Church St & 28th St,,37.745398,-122.426955,,https://SFMTA.com/18158


In [32]:
# Filter stops based on the provided stop IDs
filtered_stops1 = stop[stop['stop_id'].isin(unique_stop_ids1)]

In [34]:
# Extract the latitude and longitude for each stop ID
stop_locations = filtered_stops1[['stop_id', 'stop_lat', 'stop_lon']]

In [35]:
stop_locations

,stop_id,stop_lat,stop_lon
0,390,37.721190,-122.475153
3,3018,37.727859,-122.402994
5,3020,37.726670,-122.407460
7,3023,37.790249,-122.482263
10,3032,37.777396,-122.461751
...,...,...,...
3254,8116,37.794860,-122.408009
3261,8135,37.783610,-122.485079
3268,8152,37.803565,-122.458967
3269,8153,37.802778,-122.461552
